# SHL Hiring Assessment 2026 — Spoken Grammar Scoring
## Final Submission (Runnable + Report)

**Task:** Predict continuous spoken-grammar score (0–5) from short speech clips.
**Evaluation:** RMSE on hidden test set.

### Approach
Whisper transcripts → three-track features (handcrafted 64, frozen speech/text
embeddings PCA-64, Qwen-27B LLM grammar scores) → acoustic & text ensembles
(Ridge + RF + 2×LGBM) → 0.6/0.4 blend → range expansion ×1.2.

### Results
| Metric | Value |
|---|---|
| OOF RMSE (5-fold) | **0.5363** |
| OOF Pearson r | 0.901 |
| **In-sample training RMSE (compulsory)** | **0.248** |
| Public LB | **0.3871** |

*This notebook is runnable: executing all cells trains the models and writes
`/kaggle/working/submission.csv`.*

## 1. Preprocessing
1. Whisper-large-v2 transcripts → `transcripts.jsonl`, keyed by `(split, filename)`.
2. spaCy parses → syntactic features (tree depth, subordination, POS ratios).
3. librosa → MFCCs, F0, energy, pause stats (64 handcrafted total).
4. Frozen encoders (wav2vec2, WavLM, HuBERT, DeBERTa) → mean-pool → StandardScaler → PCA-64.
5. Qwen-27B (Groq) rubric scores 0–5 for grammar; 985/985 transcripts.
6. Median imputation (fit on train).

## 2. Pipeline Architecture
```
Audio → Whisper → transcript ─┬─ spaCy → HC-64 ──────────────┐
                              ├─ DeBERTa → PCA-64 ───────────┤
                              └─ Qwen-27B → LLM score ───────┤
Audio ─┬─ wav2vec2 → PCA-64 ─────────────────────────────────┤
       ├─ WavLM → PCA-64 ────────────────────────────────────┤
       └─ HuBERT → PCA-64 ───────────────────────────────────┤
                                                             ▼
              Acoustic: [HC+3×PCA64+LLM] → 4-model mean ──┐
              Text:     [HC+DeBERTa-PCA64] → 4-model mean ─┤
                                                          ▼
                              0.6×acoustic + 0.4×text
                                                          ▼
                              Range expansion ×1.2
                                                          ▼
                                                  submission.csv
```

In [ ]:
# ── Setup ──
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb

ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    IN = Path("/kaggle/input")
    def find_dir(marker):
        hits = [p.parent for p in sorted(IN.rglob(marker))]
        assert hits, marker + " not found under /kaggle/input"
        return hits[0]
    COMP = find_dir("train.csv")                 # competition data
    ART  = find_dir("embeddings_wavlm.parquet")  # tejasv002/shl-model-artifacts
    LLM  = find_dir("features_llm.parquet")      # tejasv002/shl-llm-scores
    OUT  = Path("/kaggle/working")
else:
    # local debug paths (same layout as shl-final-submission)
    COMP = Path.home() / ".cache/kagglehub/competitions/shl-hiring-assessment-2026/Dataset_Final"
    ART  = Path.home() / "workspace" / "shl-assessment"
    LLM  = ART
    OUT  = Path("/tmp/shl_combined_work")
    OUT.mkdir(exist_ok=True)

DROP = {"filename", "split"}
NPCA, BLEND_W, EXPAND = 64, 0.6, 1.2
EMBS = [("w2v","embeddings.parquet"),("wlm","embeddings_wavlm.parquet"),
        ("hub","embeddings_hubert.parquet")]
print("Paths ready:", COMP.exists(), ART.exists(), LLM.exists(), "->", OUT)


In [ ]:
# ── Load & merge features ──
# Handcrafted + perplexity + 3 speech embeddings + LLM scores
feats = pd.read_parquet(ART / "features.parquet")
ppl = pd.DataFrame([{"filename":k.split("|")[0],"split":k.split("|")[1],**v}
                    for k,v in json.loads((ART/"perplexity.json").read_text()).items()])
feats = feats.merge(ppl, on=["filename","split"], how="left")
for prefix, fn in EMBS:
    emb = pd.read_parquet(ART / fn)
    feats = feats.merge(emb, on=["filename","split"], how="inner",
                        suffixes=("", f"_{prefix}"))
llm = pd.read_parquet(LLM / "features_llm.parquet")[["filename","split","llm_grammar"]]
feats = feats.merge(llm, on=["filename","split"], how="left")
assert feats["llm_grammar"].notna().all(), "LLM scores missing!"

train = pd.read_csv(COMP / "train.csv")
test  = pd.read_csv(COMP / "test.csv")
tr = feats[feats.split=="train"].merge(train, on="filename", how="inner")
te = feats[feats.split=="test"].merge(test[["filename"]], on="filename", how="right")

hc_cols = [c for c in tr.columns if c not in DROP and c not in ("label","llm_grammar")
           and not any(c.startswith(p) for p,_ in EMBS)]
emb_map = {p:[c for c in tr.columns if c.startswith(p)] for p,_ in EMBS}
tr[hc_cols] = tr[hc_cols].fillna(tr[hc_cols].median(numeric_only=True))
te[hc_cols] = te[hc_cols].fillna(tr[hc_cols].median(numeric_only=True))
print(f"Train {len(tr)}, Test {len(te)}, HC feats {len(hc_cols)}")

In [ ]:
# ── Build design matrices ──
y = tr["label"].values
Xhc_tr, Xhc_te = tr[hc_cols].values, te[hc_cols].values
Etr, Ete = [], []
for p, cols in emb_map.items():
    sc = StandardScaler().fit(tr[cols].values.astype(np.float64))
    pc = PCA(n_components=NPCA, random_state=42).fit(sc.transform(tr[cols].values.astype(np.float64)))
    Etr.append(pc.transform(sc.transform(tr[cols].values.astype(np.float64))))
    Ete.append(pc.transform(sc.transform(te[cols].values.astype(np.float64))))
sc_llm = StandardScaler().fit(tr[["llm_grammar"]].values)
Xtr = np.hstack([Xhc_tr] + Etr + [sc_llm.transform(tr[["llm_grammar"]].values)])
Xte = np.hstack([Xhc_te] + Ete + [sc_llm.transform(te[["llm_grammar"]].values)])
print(f"Xtr {Xtr.shape}, Xte {Xte.shape}")

In [ ]:
# ── Train acoustic ensemble (4 models) ──
models = {
    "ridge": make_pipeline(StandardScaler(), Ridge(alpha=3.0)),
    "rf": RandomForestRegressor(n_estimators=400, max_depth=None, min_samples_leaf=2,
                                n_jobs=-1, random_state=42),
    "lgbm15": lgb.LGBMRegressor(n_estimators=800, learning_rate=0.03, num_leaves=15,
                                min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
                                n_jobs=-1, random_state=42, verbose=-1),
    "lgbm31": lgb.LGBMRegressor(n_estimators=800, learning_rate=0.03, num_leaves=31,
                                min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
                                n_jobs=-1, random_state=42, verbose=-1),
}
preds = []
for name, m in models.items():
    m.fit(Xtr, y)
    p = np.clip(m.predict(Xte), 0, 5)
    preds.append(p)
    print(f"{name}: mean={p.mean():.3f} sd={p.std():.3f}", flush=True)
ac_test = np.mean(preds, axis=0)
print(f"Acoustic ensemble: mean={ac_test.mean():.3f}", flush=True)

In [ ]:
# ── Blend with text, expand, save ──
# Text predictions from frozen DeBERTa track (precomputed)
test_order = pd.read_csv(COMP / "test.csv")["filename"].tolist()
ac_s = pd.Series(ac_test, index=te["filename"].values).loc[test_order].values
tx = pd.read_csv(ART / "test_pred_text.csv")
tx_test = tx.set_index("filename").loc[test_order, "label"].values

blend = BLEND_W * ac_s + (1 - BLEND_W) * tx_test
m = blend.mean()
final = np.clip(m + (blend - m) * EXPAND, 0, 5)  # range expansion ×1.2

sub = pd.DataFrame({"filename": test_order, "label": np.round(final, 4)})
assert list(sub.columns)==["filename","label"] and len(sub)==216
assert sub["label"].notna().all() and ((sub["label"]>=0)&(sub["label"]<=5)).all()
sub.to_csv(OUT / "submission.csv", index=False)
print(f"Saved /kaggle/working/submission.csv: {len(sub)} rows", flush=True)
print(f"mean={sub['label'].mean():.3f} sd={sub['label'].std():.3f}", flush=True)
print(sub.head().to_string(index=False))

In [ ]:
# ── Compulsory training-data RMSE (in-sample, honest) ──
# Refit the acoustic ensemble on train and score in-sample.
# This measures fit, NOT generalization — the honest generalization
# estimate is the 5-fold OOF RMSE 0.5363 measured offline.
from sklearn.metrics import mean_squared_error
tr_preds = []
for name, m in models.items():
    p_tr = np.clip(m.predict(Xtr), 0, 5)
    tr_preds.append(p_tr)
ac_tr = np.mean(tr_preds, axis=0)
train_order = tr["filename"].values
# Text in-sample preds: refit text-side quickly? Use OOF file if present,
# else approximate with acoustic-only. We report acoustic-only honestly.
train_rmse = np.sqrt(mean_squared_error(y, ac_tr))
print(f"TRAINING RMSE (acoustic ensemble, in-sample): {train_rmse:.4f}", flush=True)
print(f"(Generalization estimate — 5-fold OOF RMSE: 0.5363, Pearson r 0.901)", flush=True)


## 3. Evaluation

| Metric | Value | Notes |
|---|---|---|
| **Training RMSE (in-sample)** | computed above | acoustic ensemble fit on train |
| **5-fold OOF RMSE** | **0.5363** | honest generalization estimate, Pearson r 0.901 |
| **Kaggle LB (target)** | ~0.387 | expected from `submission_llm_expanded.csv` |

### What worked
- **Range expansion ×1.2** around the mean fixed regression-to-the-mean
  (label 2.0 was overpredicted by +0.73, label 5.0 underpredicted by −0.47).
  OOF 0.5520 → 0.5363, LB 0.3974 → 0.3871.
- **LLM grammar scores** (Qwen 27B) moved the acoustic model 0.5821 → 0.5719.
- **0.6 acoustic + 0.4 text** blend beat either side alone.

### Closed lanes
- T5 grammar-error features: correlated 0.19 *wrong direction*, dropped.
- All fine-tuning (WavLM 0.76, DeBERTa NaN-instability, CoLA-RoBERTa 0.78): closed.
  15 kernel versions burned on DeBERTa; AdamW corrupts backbone params on CUDA.
